# HRNet（PASCAL VOC 2007を用いたセマンティックセグメンテーション）

---
## 目的
複数解像度の並列ブランチを最初から最後まで維持し続けるセマンティックセグメンテーション手法であるHRNet（High-Resolution Network）を，PASCAL VOC 2007データセットを用いて実装する．これまで扱ってきたFCN・SegNet・U-Net・PSPNetなどのエンコーダ・デコーダ構造とは異なり，高解像度ブランチを一度も手放さないままステージごとに低解像度ブランチを追加し，各ステージで解像度間の情報交換（fusion）を繰り返すことで高解像度な特徴表現を保ち続ける設計思想を理解する．

## モジュールのインポート
`torchmetrics`は標準ではインストールされていないため，はじめに`pip install`でインストールしたのち，必要なモジュールをインポートし，GPUを使用した計算が可能かどうかを確認する．HRNetは事前学習済みバックボーンを使わずスクラッチで実装するため，`torchvision.models`は使用しない．

In [ ]:
!pip install -q torchmetrics

import os
import zipfile

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.datasets import VOCSegmentation
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassJaccardIndex
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## データセットの読み込みと確認
PASCAL VOC 2007データセットを読み込みます．20クラスの物体に対して画素単位のクラスラベルが付与されたデータセットで，学習用（trainval）422枚，評価用（test）210枚の画像から構成されます．画像枚数が少ない理由やデータセットの詳細は`fcn.ipynb`および`02_dnn_simple_pytorch/existing_dataset.ipynb`を参照してください．

ここでは，Googleドライブに配置したVOCdevkit（2007）のzipファイルを`gdown`でダウンロードし，`torchvision.datasets.VOCSegmentation`で読み込みます．

In [ ]:
VOC_CLASSES = [
    'background', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
n_class = len(VOC_CLASSES)  # 21（背景を含む）
IGNORE_INDEX = 255  # 物体境界などの無視領域
IMG_SIZE = 256

voc_root = './'
devkit_dir = os.path.join(voc_root, 'VOCdevkit')
if not os.path.isdir(devkit_dir):
    gdown.download(id='1xOO8ViO8NPyPngm24POj5a49zZOSkkMg', output='VOCdevkit.zip', quiet=False)
    with zipfile.ZipFile('VOCdevkit.zip') as f:
        f.extractall(voc_root)

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


class VOCSegmentationDataset(torch.utils.data.Dataset):
    """VOCSegmentationをラップし，画像とマスクを同じサイズにリサイズしたTensorとして返すDataset"""
    def __init__(self, image_set, img_size=IMG_SIZE):
        self.voc = VOCSegmentation(root=voc_root, year='2007', image_set=image_set, download=False)
        self.img_size = img_size
        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.voc)

    def __getitem__(self, idx):
        image, mask = self.voc[idx]
        image = self.img_transform(image)
        # マスクは値がそのままクラスIDのため，補間で値が混ざらないよう最近傍補間でリサイズする
        mask = TF.resize(mask, [self.img_size, self.img_size], interpolation=TF.InterpolationMode.NEAREST)
        mask = torch.from_numpy(np.array(mask, dtype=np.int64))
        return image, mask


batch_size = 8
train_data = VOCSegmentationDataset('trainval')
test_data = VOCSegmentationDataset('test')
print('train:', len(train_data), 'test:', len(test_data))

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

image, mask = train_data[0]
print('image:', image.shape, 'mask:', mask.shape, 'mask unique:', torch.unique(mask))

## VOCカラーパレットと可視化用関数
クラスIDの2次元配列（マスク）をそのまま画像として表示しても分かりにくいため，各クラスIDに固有の色を割り当ててカラー画像に変換する関数を用意します．ここでは，VOCdevkitで標準的に使われているビット演算によるカラーパレットの生成方法を使用します．

In [ ]:
def voc_colormap(n=n_class):
    """VOCdevkitで標準的に使われる、クラスIDから固有のRGB色を生成する関数"""
    cmap = np.zeros((n, 3), dtype=np.uint8)
    for i in range(n):
        r = g = b = 0
        c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j)
            g |= ((c >> 1) & 1) << (7 - j)
            b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        cmap[i] = [r, g, b]
    return cmap


VOC_COLORMAP = voc_colormap()


def decode_segmap(mask):
    """クラスIDのマスク(H, W)を，可視化用のカラー画像(H, W, 3)に変換する"""
    mask = mask.clone()
    mask[mask == IGNORE_INDEX] = 0  # 可視化のため，無視領域は背景色として扱う
    return VOC_COLORMAP[mask.cpu().numpy()]

## HRNet（High-Resolution Network）
本ノートブックで扱うHRNetは，これまでのFCN・SegNet・U-Net・PSPNetなどとは根本的に異なる設計思想を持ちます．これまでのモデルは，バックボーンで解像度をstride 32程度まで段階的にダウンサンプリングして「単一の低解像度・高チャンネルのボトルネック特徴」を作ったのち，デコーダやスキップ接続でアップサンプリングして元の解像度に戻す，というエンコーダ・デコーダ構造を採用していました．

これに対しHRNetは，**ネットワークの最初から最後まで，高解像度のブランチ（stride 4）を一度も手放さない**まま，ステージが進むごとにより低解像度のブランチ（stride 8, 16, ...）を並列に追加していきます．そして各ステージの終わりで，すべての解像度ブランチ間で特徴を交換する「fusion（融合）」を繰り返すことで，高解像度ブランチも低解像度ブランチが持つ広い受容野の文脈情報を取り込みつつ，空間的に精細な情報を保持し続けます．姿勢推定やセグメンテーションのように画素・キーポイント単位の精度が求められるタスクで有効とされる所以です．

まずは，ネットワークの入り口となるstem（通常のResNetと同様，stride 4まで畳み込みで解像度を落とす部分）と，単一解像度のままResNetのBottleneckブロックを重ねるStage1を実装します．

In [ ]:
class BasicBlock(nn.Module):
    """3x3畳み込み×2＋residual接続を持つ基本ブロック（stride・チャンネル数は変更しない）"""
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class Bottleneck(nn.Module):
    """1x1->3x3->1x1のResNet Bottleneckブロック（出力チャンネルはplanesの4倍）"""
    expansion = 4

    def __init__(self, in_channels, planes, downsample=None):
        super().__init__()
        out_channels = planes * self.expansion
        self.conv1 = nn.Conv2d(in_channels, planes, kernel_size=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, out_channels, kernel_size=1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)


def make_stage1(in_channels=64, planes=64, blocks=2):
    """stride 4のまま，Bottleneckブロックを重ねるStage1（通常のResNetの最初のstageと同じ構成）"""
    downsample = nn.Sequential(
        nn.Conv2d(in_channels, planes * Bottleneck.expansion, kernel_size=1, bias=False),
        nn.BatchNorm2d(planes * Bottleneck.expansion),
    )
    layers = [Bottleneck(in_channels, planes, downsample=downsample)]
    for _ in range(blocks - 1):
        layers.append(Bottleneck(planes * Bottleneck.expansion, planes))
    return nn.Sequential(*layers)


_stem = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
    nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
).to(device)  # stride 4, channels 64
_stage1 = make_stage1(in_channels=64, planes=64, blocks=2).to(device)  # stride 4のまま, channels 256

# 出力ストライドが4（256 / 4 = 64）になっていることを形状で確認する
_x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(device)
_r1 = _stage1(_stem(_x))
print('stem+stage1 output shape:', _r1.shape)
assert _r1.shape[-1] == IMG_SIZE // 4
del _stem, _stage1, _x, _r1

### Transition層とFusionモジュール
ステージが進むごとに新しい低解像度ブランチを追加する処理を**Transition層**と呼びます．既存の各ブランチはチャンネル数を目的の幅に合わせる3x3畳み込みで変換し（チャンネル数が変わらない場合はそのまま），新規に追加するブランチは，直前の最も低解像度のブランチにstride 2の3x3畳み込みを適用することで生成します．

一方，各ステージ内で複数解像度のブランチ間で情報をやり取りするのが**Fusionモジュール**です．出力ブランチ$i$は，自分自身の特徴に加え，

- 自分より低解像度のブランチ$j$：1x1畳み込みでチャンネル数を合わせたのち，バイリニア補間で解像度$i$までアップサンプリング
- 自分より高解像度のブランチ$j$：解像度差の回数だけstride 2の3x3畳み込みを繰り返し適用してダウンサンプリング

をすべて加算し，最後にReLUを適用したものです．このfusionを各ステージの末尾で繰り返すことで，どのブランチも他の全解像度の情報を取り込みながら，自身の解像度を保ち続けます．

In [ ]:
class TransitionLayer(nn.Module):
    """ブランチ数・チャンネル幅の異なる構成へ変換する．ブランチ数が増える場合，新規ブランチは
    直前の最も低解像度な既存ブランチからstride 2の畳み込みで生成する．"""
    def __init__(self, in_channels_list, out_channels_list):
        super().__init__()
        num_in = len(in_channels_list)
        self.num_in = num_in
        self.transforms = nn.ModuleList()
        for i, out_channels in enumerate(out_channels_list):
            if i < num_in:
                if in_channels_list[i] != out_channels:
                    self.transforms.append(nn.Sequential(
                        nn.Conv2d(in_channels_list[i], out_channels, kernel_size=3, padding=1, bias=False),
                        nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
                    ))
                else:
                    self.transforms.append(nn.Identity())
            else:
                self.transforms.append(nn.Sequential(
                    nn.Conv2d(in_channels_list[-1], out_channels, kernel_size=3, stride=2, padding=1, bias=False),
                    nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
                ))

    def forward(self, branches):
        outputs = []
        for i, transform in enumerate(self.transforms):
            src = branches[i] if i < self.num_in else branches[-1]
            outputs.append(transform(src))
        return outputs


class FusionModule(nn.Module):
    """複数解像度ブランチ間で情報を交換するモジュール．出力ブランチiは，全ブランチjを
    解像度iに変換した特徴の総和にReLUを適用したもの．"""
    def __init__(self, channels_list):
        super().__init__()
        n = len(channels_list)
        self.n = n
        self.transforms = nn.ModuleList()
        for i in range(n):
            row = nn.ModuleList()
            for j in range(n):
                if j == i:
                    row.append(nn.Identity())
                elif j > i:
                    # 低解像度ブランチj -> 高解像度i：1x1畳み込みでチャンネルを揃えてからアップサンプリング
                    row.append(nn.Sequential(
                        nn.Conv2d(channels_list[j], channels_list[i], kernel_size=1, bias=False),
                        nn.BatchNorm2d(channels_list[i]),
                    ))
                else:
                    # 高解像度ブランチj -> 低解像度i：(i-j)回のstride 2畳み込みで段階的にダウンサンプリング
                    layers = []
                    for k in range(i - j):
                        last = (k == i - j - 1)
                        out_ch = channels_list[i] if last else channels_list[j]
                        layers.append(nn.Conv2d(channels_list[j], out_ch, kernel_size=3, stride=2, padding=1, bias=False))
                        layers.append(nn.BatchNorm2d(out_ch))
                        if not last:
                            layers.append(nn.ReLU(inplace=True))
                    row.append(nn.Sequential(*layers))
            self.transforms.append(row)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, branches):
        target_sizes = [b.shape[-2:] for b in branches]
        outputs = []
        for i in range(self.n):
            fused = 0
            for j in range(self.n):
                t = self.transforms[i][j](branches[j])
                if j > i:
                    t = F.interpolate(t, size=target_sizes[i], mode='bilinear', align_corners=False)
                fused = fused + t
            outputs.append(self.relu(fused))
        return outputs

### Stage2・Stage3：並列ブランチの構築
Stage2では，Stage1が出力した単一の高解像度特徴（stride 4, 256ch）から，Transition層によってstride 4（width 32）とstride 8（width 64）の2つのブランチを作ります．各ブランチにBasicBlockを適用したのち，Fusionモジュールで一度情報交換を行います．

Stage3では，さらにTransition層でstride 16（width 128）のブランチを追加し，3つのブランチに対して「BasicBlock適用→Fusion」を2回繰り返します（計算量の都合上，オリジナル論文のStage4・4つ目のブランチ（stride 32）は省略しています）．

この「BasicBlock適用→Fusion」という処理単位をHRStageとしてまとめて実装します．

In [ ]:
class HRStage(nn.Module):
    """各ブランチにBasicBlockを適用したのち，Fusionモジュールで情報交換を行う処理をnum_modules回繰り返す"""
    def __init__(self, channels_list, num_blocks=2, num_modules=1):
        super().__init__()
        self.branch_blocks = nn.ModuleList([
            nn.ModuleList([
                nn.Sequential(*[BasicBlock(ch) for _ in range(num_blocks)])
                for ch in channels_list
            ])
            for _ in range(num_modules)
        ])
        self.fusions = nn.ModuleList([FusionModule(channels_list) for _ in range(num_modules)])

    def forward(self, branches):
        for blocks, fusion in zip(self.branch_blocks, self.fusions):
            branches = [block(b) for block, b in zip(blocks, branches)]
            branches = fusion(branches)
        return branches

### 予測ヘッドとHRNet全体
最終的な予測ヘッドでは，Stage3が出力した3つの解像度ブランチ（stride 4/8/16）をすべて最も高解像度なstride 4のブランチのサイズにバイリニア補間でアップサンプリングし，チャンネル方向にconcatします．1x1畳み込みでチャンネルを整えたのち，入力画像と同じ解像度までさらにアップサンプリング（stride 4→stride 1）し，最後に1x1畳み込みでクラス数チャンネルに変換します．

これまでの各パーツ（stem, Stage1, Transition, Stage2・3, 予測ヘッド）を組み合わせて，HRNet全体を1つの`nn.Module`として実装します．

In [ ]:
class HRNet(nn.Module):
    def __init__(self, n_class=n_class):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
        )  # stride 4, channels 64
        self.stage1 = make_stage1(in_channels=64, planes=64, blocks=2)  # stride 4のまま, channels 256

        self.transition1 = TransitionLayer([256], [32, 64])       # R1: stride 4/width32, R2: stride 8/width64
        self.stage2 = HRStage([32, 64], num_blocks=2, num_modules=1)

        self.transition2 = TransitionLayer([32, 64], [32, 64, 128])  # R3: stride 16/width128を追加
        self.stage3 = HRStage([32, 64, 128], num_blocks=2, num_modules=2)

        fused_channels = 32 + 64 + 128
        self.fuse_conv = nn.Sequential(
            nn.Conv2d(fused_channels, 128, kernel_size=1, bias=False), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(128, n_class, kernel_size=1)

    def forward(self, x):
        input_size = x.shape[-2:]
        x = self.stem(x)
        x = self.stage1(x)

        branches = self.transition1([x])
        branches = self.stage2(branches)

        branches = self.transition2(branches)
        branches = self.stage3(branches)  # [stride4/32ch, stride8/64ch, stride16/128ch]

        # 全ブランチを最も高解像度（stride4）のサイズへアップサンプリングしてconcat
        target_size = branches[0].shape[-2:]
        upsampled = [branches[0]] + [
            F.interpolate(b, size=target_size, mode='bilinear', align_corners=False) for b in branches[1:]
        ]
        fused = self.fuse_conv(torch.cat(upsampled, dim=1))

        out = F.interpolate(fused, size=input_size, mode='bilinear', align_corners=False)  # stride4 -> stride1
        return self.classifier(out)

## 損失関数
セグメンテーションは，画素ごとのクラス分類問題とみなせるため，`nn.CrossEntropyLoss`をそのまま利用できます．マスクに含まれる無視領域（ラベル255）は`ignore_index`で損失計算から除外します．

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

## ネットワークの作成
HRNetを構築し，最適化手法としてAdamを設定します．事前学習済み重みを持たないスクラッチ学習のため，事前学習済みバックボーンを使うモデル（FCNやPSPNetなど）よりも大きい学習率を用います．

In [ ]:
model = HRNet(n_class=n_class).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

num_params = sum(p.numel() for p in model.parameters())
print('パラメータ数:', num_params)

## 学習
学習データ（trainval）を用いてHRNetを学習します．VOC2007のセグメンテーション用データは422枚と少ないため，`epoch_num`を多めに設定しています．

In [ ]:
epoch_num = 30

model.train()
for epoch in range(epoch_num):
    sum_loss = 0.0
    count = 0
    for image, mask in train_loader:
        image, mask = image.to(device), mask.to(device)

        optimizer.zero_grad()
        output = model(image)
        loss = criterion(output, mask)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * image.size(0)
        count += image.size(0)

    print(f'epoch: {epoch + 1}, mean loss: {sum_loss / count:.4f}')

## テスト（mIoU評価）
評価用データ（test）に対して，クラスごとのIoU（Intersection over Union）の平均であるmIoU（mean IoU）を求めます．mIoUの算出には，`torchmetrics.classification.MulticlassJaccardIndex`を使用し，無視領域（ラベル255）は`ignore_index`で評価から除外します．

In [ ]:
metric = MulticlassJaccardIndex(num_classes=n_class, ignore_index=IGNORE_INDEX, average='macro').to(device)

model.eval()
with torch.no_grad():
    for image, mask in test_loader:
        image, mask = image.to(device), mask.to(device)
        output = model(image)
        pred = output.argmax(dim=1)
        metric.update(pred, mask)

print(f'mIoU: {metric.compute().item():.4f}')

## セグメンテーション結果の可視化
評価用データから1枚取り出し，入力画像・正解マスク・予測マスクを並べて表示します．

In [ ]:
model.eval()
image, mask = test_data[0]
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    pred = output.argmax(dim=1).squeeze(0).cpu()

# 表示用に正規化前の画素値へ戻す
mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
image_vis = (image * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image_vis); axes[0].set_title('input'); axes[0].axis('off')
axes[1].imshow(decode_segmap(mask)); axes[1].set_title('ground truth'); axes[1].axis('off')
axes[2].imshow(decode_segmap(pred)); axes[2].set_title('prediction'); axes[2].axis('off')
plt.show()

## オリジナルのHRNetとの違い

| 項目 | オリジナル論文 (Wang et al., 2019, 例：HRNetV2-W48) | 本ノートブック |
| :-- | :-- | :-- |
| 解像度ブランチ数 | 4段階（stride 4/8/16/32）を最終ステージまで維持 | 3段階（stride 4/8/16）で打ち切り，stride 32のブランチは追加しない |
| チャンネル幅 | 48/96/192/384（W48の場合）など比較的大きい | 32/64/128 まで縮小した軽量構成 |
| 各ステージのブロック数 | 各ステージで複数モジュール×複数ブロック（例：Stage4は3モジュール×4ブロック） | 各モジュール2ブロックのみ．Stage2は1モジュール，Stage3は2モジュールに削減 |
| 事前学習済み重み | ImageNet事前学習済みのHRNetV2-W48などを使用するのが一般的 | 独自に縮小した構成のため対応する事前学習済み重みが存在せず，スクラッチから学習 |
| 予測ヘッド | HRNetV2は全ブランチをconcatし1x1畳み込みで出力（本実装と同様） | 同様に全ブランチconcat方式（HRNetV2方式）を採用 |

## 課題

1. 各Stageの`num_blocks`・`num_modules`を増やす，またはブランチ数を4つに増やしてstride 32のブランチを追加した場合に，パラメータ数・学習時間・mIoUがどのように変化するか確認してください．
2. `channels_list`の幅（32/64/128）を変更し，パラメータ数とmIoUの関係を確認してください．
3. Stage2・Stage3のFusionモジュールを取り除き（各ブランチを独立に学習させ，最後の予測ヘッドでのみconcatする）場合と比較し，ステージ内で繰り返し情報交換を行うことの効果を確認してください．